In [1]:
from datasets.batch_integration import BatchIntDataset
from utils.experiment_utils import get_all_experiments_info, load_best_model
from utils.eval_utils import compute_mmd_distance, compute_sw_distance
import os

import scvi
import scanpy as sc
import pandas as pd
from collections import defaultdict
from sklearn.neighbors import NearestNeighbors

import hydra
from omegaconf import OmegaConf

import torch
import numpy as np
from geomloss import SamplesLoss

import harmonypy as hm

# silence all warnings
import warnings
warnings.filterwarnings("ignore")

/orcd/home/002/gokulg/miniforge3/envs/gde/lib/python3.11/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/orcd/home/002/gokulg/miniforge3/envs/gde/lib/python3.11/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/orcd/home/002/gokulg/miniforge3/envs/gde/lib/python3.11/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [2]:
n_pcs = 10
train_set = BatchIntDataset(
    root="data",
    split="train",
    n_pcs=n_pcs
)

test_set = BatchIntDataset(
    root="data",
    split="test",
    n_pcs=n_pcs
)

train_donors = train_set.donors
test_donors = test_set.donors
print(train_donors)
print(test_donors)

data loaded !
n train donors: 53
n test donors: 3
data loaded !
n train donors: 53
n test donors: 3
['mouse_pancreatic_islet_atlas_Hrovatin__Fltp_P16__145_mGFP', 'mouse_pancreatic_islet_atlas_Hrovatin__Fltp_P16__146_mRFP', 'mouse_pancreatic_islet_atlas_Hrovatin__Fltp_P16__147_mTmG', 'mouse_pancreatic_islet_atlas_Hrovatin__Fltp_adult__mouse1', 'mouse_pancreatic_islet_atlas_Hrovatin__Fltp_adult__mouse2', 'mouse_pancreatic_islet_atlas_Hrovatin__Fltp_adult__mouse3', 'mouse_pancreatic_islet_atlas_Hrovatin__Fltp_adult__mouse4', 'mouse_pancreatic_islet_atlas_Hrovatin__NOD__SRR10985097', 'mouse_pancreatic_islet_atlas_Hrovatin__NOD__SRR10985098', 'mouse_pancreatic_islet_atlas_Hrovatin__NOD__SRR10985099', 'mouse_pancreatic_islet_atlas_Hrovatin__NOD_elimination__SRR7610295', 'mouse_pancreatic_islet_atlas_Hrovatin__NOD_elimination__SRR7610296', 'mouse_pancreatic_islet_atlas_Hrovatin__NOD_elimination__SRR7610297', 'mouse_pancreatic_islet_atlas_Hrovatin__NOD_elimination__SRR7610298', 'mouse_pancreat

In [3]:
configs = get_all_experiments_info('/orcd/home/002/gokulg/orcd/scratch/CoupledDistributionEmbeddings/outputs/', False)

cfgs = [c for c in configs if 'batchint' in c['name']]

energy_models = [c for c in cfgs if 'mmd' in c['config']['generator'].values()
                 and 'onehot' not in c['name']]

encoders = [c['config']['encoder'] for c in energy_models]
                
print("energy models:\n")
for model in energy_models:
    print(model['name'])
    print(model['config']['encoder']['_target_'])
    print()

energy models:

batchint_51fd5cc1e1cb76538d5e871f133014cc
encoder.encoders.DistributionEncoderGNN

batchint_f12c45b6bdf2fd54506f8c4969f754da
encoder.kernel_mean.KMEEncoder

batchint_f7f974710ed71ca9bf215bd404b15b20
encoder.encoders.DistributionEncoderResNetTx



In [4]:
def load_model(cfg, path, device):
    enc = hydra.utils.instantiate(cfg['encoder'])
    gen = hydra.utils.instantiate(cfg['generator'])
    state = load_best_model(path)
    enc.load_state_dict(state['encoder_state_dict'])
    gen.load_state_dict(state['generator_state_dict'])
    enc.eval()
    gen.eval()
    enc.to(device)
    gen.to(device)
    return enc, gen

In [5]:
results = {
    'encoder' : [],
    'energy' : [],
    'mmd' : [],
    'sw' : []
}

device = 'cuda'
n_samples = 10

energy = SamplesLoss('energy')

for model in energy_models:
    print(f"Evaluating model: {model['name']}")
    encoder, generator = load_model(model['config'], model['dir'], 'cuda')
    
    for p in range(len(test_set)):
        for _ in range(n_samples):

            batch = test_set[p]

            source_samples = batch['source_samples'].to(device).unsqueeze(0)
            target_samples = batch['target_samples'].to(device).unsqueeze(0)
            
            with torch.no_grad():
                source_latent = encoder(source_samples)
                target_latent = encoder(target_samples)
                samples = generator.sample(source_samples.reshape(-1, n_pcs), source_latent, target_latent)
        
            # Compute all metrics
            results['energy'].append(energy(samples.squeeze(0), target_samples.squeeze(0)).item())
            results['mmd'].append(compute_mmd_distance(samples, target_samples).item())
            results['sw'].append(compute_sw_distance(samples, target_samples).item())
            results['encoder'].append(model['config']['encoder']['_target_'])

Evaluating model: batchint_51fd5cc1e1cb76538d5e871f133014cc
Evaluating model: batchint_f12c45b6bdf2fd54506f8c4969f754da
Evaluating model: batchint_f7f974710ed71ca9bf215bd404b15b20


In [6]:
results_df = pd.DataFrame(results)
print('mean')
print(results_df.groupby('encoder').mean())
print('\n\n')
print('sem')
print(results_df.groupby('encoder').std()/np.sqrt(results_df.groupby('encoder').size().values))

mean
                                                energy       mmd        sw
encoder                                                                   
encoder.encoders.DistributionEncoderGNN       0.048565  0.097132  0.310763
encoder.encoders.DistributionEncoderResNetTx  0.044868  0.089737  0.311700
encoder.kernel_mean.KMEEncoder                0.690532  1.381065  0.728566



sem
                                                energy       mmd        sw
encoder                                                                   
encoder.encoders.DistributionEncoderGNN       0.007468  0.014935  0.039712
encoder.encoders.DistributionEncoderResNetTx  0.005311  0.010622  0.031784
encoder.kernel_mean.KMEEncoder                0.071211  0.142421  0.046475
